In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

# 1. Load the raw Kaggle dataset
print("Loading raw data...")
df = pd.read_csv('../data/raw/cs-training.csv')

# 2. Feature Engineering: Aligning with the FastAPI Contract
print("Engineering features...")
df['Income_Annual'] = df['MonthlyIncome'] * 12
df['Spending_Ratio'] = df['DebtRatio']
df['Utility_Bill_Late_Count'] = (
    df['NumberOfTime30-59DaysPastDueNotWorse'] + 
    df['NumberOfTime60-89DaysPastDueNotWorse'] + 
    df['NumberOfTimes90DaysLate']
)
df['Credit_History_Length_Months'] = (df['age'] * 12) - 216
df.loc[df['Credit_History_Length_Months'] < 0, 'Credit_History_Length_Months'] = 0

# --- THE FIX: Breaking Perfect Colinearity ---
# We inject realistic variance. Some save 0x their income, some save up to 6x.
# Seed 42 guarantees your demo data remains identical every time you run this.
np.random.seed(42)
savings_multipliers = np.random.uniform(0.0, 6.0, size=len(df))
df['Savings_Balance'] = df['MonthlyIncome'] * savings_multipliers
# ---------------------------------------------

df['Default'] = df['SeriousDlqin2yrs']

# 3. Handle missing values properly (NO DROPNA)
print("Imputing missing values with medians...")
api_features = [
    'Income_Annual', 'Savings_Balance', 'Spending_Ratio', 
    'Utility_Bill_Late_Count', 'Credit_History_Length_Months', 'Default'
]

# Create the clean dataframe with the selected columns
df_clean = df[api_features].copy()

# Initialize imputer for median strategy and apply it
imputer = SimpleImputer(strategy='median')
df_clean[api_features] = imputer.fit_transform(df_clean[api_features])

# 4. Outlier Capping
print("Capping massive outliers at the 99th percentile...")
outlier_columns = ['Income_Annual', 'Spending_Ratio', 'Savings_Balance']

for col in outlier_columns:
    upper_limit = df_clean[col].quantile(0.99)
    df_clean[col] = df_clean[col].clip(upper=upper_limit)

# 5. Export the perfectly cleaned & capped dataset
print(f"Success! {len(df_clean)} API-compliant records ready.")
df_clean.to_csv('../data/processed/clean_features.csv', index=False)